# Kaggle — YOLO đếm xe trên 5 dataset Hugging Face

1. Bật GPU + Internet, thêm Secret HF_TOKEN quyền đọc cả 5 repo và cấp notebook truy cập.
2. Add Input **best.pt mới sau fine-tune** (Kaggle Dataset private hoặc Output notebook train).
3. Sửa WEIGHTS sau cell liệt kê file. Kiểm tra class mapping.
4. Mặc định benchmark 1.000 ảnh tổng cộng, không phải 1.000/repo. Khi ổn, LIMIT=None.
5. Save Version → Save & Run All để chạy nền; phiên này chạy lại từ đầu.

Không cần tải toàn bộ ảnh về máy. Hỗ trợ ảnh rời và Parquet có image.bytes.
Mỗi lần chỉ giữ một Parquet shard trên ổ đĩa, decode theo batch; nếu ảnh rời chỉ tải từng batch.
Repo có cả hai loại mặc định chọn Parquet để tránh đếm hai lần. ZIP chưa hỗ trợ.
Camera/timestamp phải xác định được từ cột metadata hoặc đường dẫn ảnh; lỗi schema sẽ dừng.
Không tự upload kết quả, không dùng Spark, không tự rơi xuống CPU.


In [ ]:
%pip install "ultralytics>=8.3,<8.4" "huggingface_hub>=0.23" "Pillow>=10,<13" pyarrow


In [ ]:
import os, json, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
import torch
from ultralytics import YOLO
assert sys.version_info >= (3,11)
assert torch.cuda.is_available(), 'Bật GPU trước'
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('GPU:', torch.cuda.get_device_name(0))
print('Weights có sẵn trong Input:')
for p in Path('/kaggle/input').rglob('best.pt'):
    print(p)


In [ ]:
WEIGHTS = Path('/kaggle/input/YOUR-TRAINING-OUTPUT/yolo-training/yolo11l-v20-run1/weights/best.pt')
assert WEIGHTS.is_file(), 'Sửa WEIGHTS thành đường dẫn best.pt vừa liệt kê'
model = YOLO(str(WEIGHTS))
print('Mapping:', model.names)
del model

# Mapping dataset v20: 0=2-wheel vehicles, 1=bus, 2=car, 3=truck.
# Xác nhận model.names trước khi dùng mapping này.
CONFIG = {
    'repos': [
        'ericng17/hcm_traffic_snapshots',
        'ericng17/hcm_traffic_snapshots2',
        'ericng17/hcm_traffic_snapshots3',
        'ericng17/hcm_traffic_snapshots4',
        'ericng17/hcm_traffic_snapshots5',
    ],
    'weights': str(WEIGHTS),
    'output': '/kaggle/working/vehicle-counts',
    'classes': [0,1,2,3],
    'batch': 4,
    'imgsz': 1024,
    'conf': 0.25,
    'iou': 0.7,
    'max_det': 1000,
    'half': False,
    'timezone': 'Asia/Ho_Chi_Minh',  # Phải đúng timezone crawler, đổi UTC nếu cần.
    'limit': 1000,                 # None = toàn bộ; giới hạn số ảnh MỚI mỗi lần chạy.
    'preview_count': 6,
    'workers': 4,
    'format': 'auto',              # auto / parquet / raw
    'image_column': 'image',
    'path_column': None,           # None: path / file_name / filename / image.path
    'camera_column': None,         # None: camera_id / cam_id / camera / thư mục cha
    'timestamp_column': None,      # None: timestamp / datetime / time / tên file
}


## Kiểm tra cấu trúc cả 5 repo trước khi chạy dài
Cell này chỉ liệt kê metadata, không tải ảnh. Schema Parquet được in khi đọc shard đầu tiên.

In [ ]:
api = HfApi(token=os.environ['HF_TOKEN'])
for repo in CONFIG['repos']:
    info = api.repo_info(repo, repo_type='dataset')
    files = api.list_repo_files(repo, repo_type='dataset', revision=info.sha)
    parquet = [p for p in files if p.endswith('.parquet')]
    images = [p for p in files if Path(p).suffix.lower() in {'.jpg','.jpeg','.png'}]
    print(repo, 'revision:', info.sha, 'Parquet:', len(parquet), 'Ảnh rời:', len(images))
    print('Ví dụ:', (parquet or images or files)[:3])
    assert parquet or images, 'Không tìm thấy định dạng hỗ trợ; kiểm tra repo/ZIP.'


## Engine inference và checkpoint
Cell chứa sẵn mã script, không cần upload .py riêng.

In [ ]:
ENGINE = Path('/kaggle/working/infer_hf_yolo.py')
ENGINE.write_text('"""Bounded-memory Hugging Face YOLO inference, SQLite resume, raw images/Parquet."""\nimport csv\nimport hashlib\nimport io\nimport json\nimport sqlite3\nimport tempfile\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nfrom PIL import Image\n\n\ndef identity(row, path, timezone, camera_column=None, timestamp_column=None):\n    camera_keys = [camera_column] if camera_column else [\'camera_id\', \'cam_id\', \'camera\']\n    time_keys = [timestamp_column] if timestamp_column else [\'timestamp\', \'datetime\', \'time\']\n    camera = next((row[k] for k in camera_keys if row.get(k) is not None), None)\n    timestamp = next((row[k] for k in time_keys if row.get(k) is not None), None)\n    path = Path(str(path or \'\').replace(\'\\\\\', \'/\'))\n    if camera is None and path.parent.name:\n        camera = path.parent.name\n    if timestamp is None:\n        timestamp = path.stem\n    if camera is None or isinstance(camera, (dict, list)):\n        raise ValueError(\'Cannot identify camera. Configure CAMERA_COLUMN/PATH_COLUMN.\')\n    if isinstance(timestamp, datetime):\n        dt = timestamp\n    else:\n        value = str(timestamp)\n        try:\n            dt = datetime.strptime(value, \'%Y%m%d_%H%M%S\')\n        except ValueError:\n            dt = datetime.fromisoformat(value.replace(\'Z\', \'+00:00\'))\n    if dt.tzinfo is None:\n        dt = dt.replace(tzinfo=ZoneInfo(timezone))\n    return str(camera), dt.isoformat()\n\n\ndef setup_db(output, config):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(output / \'counts.sqlite\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT)\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS completed (repo TEXT, revision TEXT, source TEXT, PRIMARY KEY(repo,revision,source))\')\n    db.execute(\'\'\'CREATE TABLE IF NOT EXISTS counts (\n        id TEXT PRIMARY KEY, repo TEXT, revision TEXT, source TEXT, row_index INTEGER,\n        camera_id TEXT, timestamp TEXT, status TEXT, total INTEGER, counts TEXT,\n        image_sha256 TEXT, error TEXT)\'\'\')\n    encoded = json.dumps(config, sort_keys=True)\n    old = db.execute("SELECT value FROM meta WHERE key=\'config\'").fetchone()\n    if old and old[0] != encoded:\n        raise ValueError(\'Configuration changed: choose a new OUTPUT directory.\')\n    db.execute("INSERT OR IGNORE INTO meta VALUES (\'config\',?)", (encoded,))\n    db.commit()\n    (output / \'run.json\').write_text(json.dumps(config, indent=2, ensure_ascii=False))\n    return db\n\n\ndef export(db, output, names):\n    target = Path(output) / \'counts.csv\'\n    tmp = target.with_suffix(\'.csv.tmp\')\n    with tmp.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'id\',\'repo\',\'revision\',\'source\',\'row_index\',\'camera_id\',\'timestamp\',\n                         \'status\',\'total_vehicles\'] + [f\'count_{k}_{v}\' for k,v in names.items()] +\n                        [\'image_sha256\',\'error\'])\n        for row in db.execute(\'SELECT * FROM counts ORDER BY repo,source,row_index\'):\n            counts = json.loads(row[9]) if row[9] else {}\n            writer.writerow(list(row[:9]) +\n                [counts.get(str(k), 0) if row[7]==\'ok\' else \'\' for k in names] + list(row[10:]))\n    tmp.replace(target)\n\n\ndef parquet_records(local, filename, config):\n    import pyarrow.parquet as pq\n    pf = pq.ParquetFile(local)\n    keys = pf.schema_arrow.names\n    image_col = config.get(\'image_column\', \'image\')\n    if image_col not in keys:\n        raise ValueError(f\'{filename}: missing image column {image_col}; columns={keys}\')\n    print(\'Parquet schema:\', pf.schema_arrow, flush=True)\n    index = 0\n    for batch in pf.iter_batches(batch_size=config[\'batch\']):\n        for row in batch.to_pylist():\n            image = row[image_col]\n            path = row.get(config.get(\'path_column\') or \'path\') or row.get(\'file_name\') or row.get(\'filename\')\n            if isinstance(image, dict):\n                path = path or image.get(\'path\')\n                content = image.get(\'bytes\')\n            elif isinstance(image, bytes):\n                content = image\n            else:\n                raise ValueError(\'Parquet image must contain embedded bytes (HF Image struct supported).\')\n            # Metadata errors are fatal: never silently lose camera/timestamp.\n            camera, timestamp = identity(row, path, config[\'timezone\'],\n                config.get(\'camera_column\'), config.get(\'timestamp_column\'))\n            yield index, camera, timestamp, content\n            index += 1\n\n\ndef run(config):\n    if config[\'batch\'] < 1 or config[\'workers\'] < 1:\n        raise ValueError(\'batch and workers must be positive\')\n    if config[\'limit\'] is not None and config[\'limit\'] < 1:\n        raise ValueError(\'limit must be positive or None\')\n    if config.get(\'format\', \'auto\') not in {\'auto\', \'raw\', \'parquet\'}:\n        raise ValueError(\'format must be auto/raw/parquet\')\n    from huggingface_hub import HfApi, hf_hub_download\n    import torch\n    import ultralytics\n    from ultralytics import YOLO\n    import os\n    assert torch.cuda.is_available(), \'Enable GPU; CPU fallback is not automatic.\'\n    weights = Path(config[\'weights\'])\n    assert weights.is_file(), f\'Weights missing: {weights}\'\n    with weights.open(\'rb\') as f:\n        weight_hash = hashlib.file_digest(f, \'sha256\').hexdigest()\n    model = YOLO(str(weights))\n    if model.task != \'detect\':\n        raise ValueError(\'Detection weights required\')\n    names = {int(k):v for k,v in model.names.items()}\n    classes = sorted(config[\'classes\'])\n    if not classes or not set(classes) <= names.keys():\n        raise ValueError(f\'Invalid classes; model mapping={names}\')\n    names = {k:names[k] for k in classes}\n    settings = {k:v for k,v in config.items() if k not in {\'weights\',\'output\',\'batch\',\'limit\',\'preview_count\',\'workers\'}}\n    settings.update(weights_sha256=weight_hash, names=names,\n                    ultralytics=ultralytics.__version__, torch=torch.__version__)\n    output = Path(config[\'output\'])\n    db = setup_db(output, settings)\n    api = HfApi(token=os.environ.get(\'HF_TOKEN\'))\n    pending, processed = [], 0\n    preview_dir = output / \'previews\'\n    preview_dir.mkdir(exist_ok=True)\n    previews = len(list(preview_dir.glob(\'*.jpg\')))\n\n    def flush():\n        nonlocal previews\n        if not pending:\n            return\n        images, good = [], []\n        for key, repo, revision, source, index, camera, timestamp, content in pending:\n            prefix = (key,repo,revision,source,index,camera,timestamp)\n            try:\n                if not content:\n                    raise ValueError(\'Missing embedded image bytes\')\n                with Image.open(io.BytesIO(content)) as image:\n                    image.load()\n                    images.append(image.convert(\'RGB\'))\n                good.append((prefix,hashlib.sha256(content).hexdigest()))\n            except (OSError, ValueError) as exc:\n                db.execute(\'INSERT OR REPLACE INTO counts VALUES (?,?,?,?,?,?,?,?,?,?,?,?)\',\n                    prefix+(\'error\',None,None,None,f\'{type(exc).__name__}: {exc}\'))\n        if images:\n            results = list(model.predict(images, device=0, imgsz=config[\'imgsz\'],\n                conf=config[\'conf\'], iou=config[\'iou\'], max_det=config[\'max_det\'],\n                classes=classes, half=config[\'half\'], stream=True, verbose=False, save=False))\n            if len(results)!=len(good):\n                raise RuntimeError(\'YOLO result count mismatch\')\n            for result,(prefix,digest) in zip(results,good):\n                counts = Counter(int(i) for i in result.boxes.cls.cpu().tolist())\n                db.execute(\'INSERT OR REPLACE INTO counts VALUES (?,?,?,?,?,?,?,?,?,?,?,?)\',\n                    prefix+(\'ok\',sum(counts.values()),json.dumps(counts),digest,None))\n                if previews < config[\'preview_count\']:\n                    Image.fromarray(result.plot()[..., ::-1]).save(preview_dir / f\'{prefix[0]}.jpg\')\n                    previews += 1\n        db.commit()\n        pending.clear()\n\n    def accept(repo, revision, source, index, camera, timestamp, content):\n        nonlocal processed\n        key = hashlib.sha256(json.dumps([repo,revision,source,index]).encode()).hexdigest()\n        if db.execute(\'SELECT 1 FROM counts WHERE id=?\',(key,)).fetchone():\n            return\n        pending.append((key,repo,revision,source,index,camera,timestamp,content))\n        processed += 1\n        if len(pending)>=config[\'batch\']:\n            flush()\n            if processed % (config[\'batch\'] * 100) == 0:\n                print(f\'Processed {processed} new images\',flush=True)\n\n    try:\n        for repo in config[\'repos\']:\n            pin_key = \'revision:\'+repo\n            old = db.execute(\'SELECT value FROM meta WHERE key=?\',(pin_key,)).fetchone()\n            revision = old[0] if old else api.repo_info(repo,repo_type=\'dataset\').sha\n            db.execute(\'INSERT OR IGNORE INTO meta VALUES (?,?)\',(pin_key,revision))\n            db.commit()\n            files = sorted(api.list_repo_files(repo,repo_type=\'dataset\',revision=revision))\n            parquet = [p for p in files if p.endswith(\'.parquet\')]\n            raw = [p for p in files if Path(p).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\'}]\n            # Prevent double counting if repo exposes both formats.\n            mode = config.get(\'format\',\'auto\')\n            use_parquet = mode==\'parquet\' or (mode==\'auto\' and bool(parquet))\n            print(repo,revision,\'parquet\' if use_parquet else \'raw\',flush=True)\n            if use_parquet:\n                if not parquet: raise ValueError(f\'{repo}: no Parquet found\')\n                for source in parquet:\n                    if db.execute(\'SELECT 1 FROM completed WHERE repo=? AND revision=? AND source=?\', (repo,revision,source)).fetchone():\n                        continue\n                    with tempfile.TemporaryDirectory(prefix=\'yolo-shard-\') as tmp:\n                        local = hf_hub_download(repo,source,repo_type=\'dataset\',revision=revision,local_dir=tmp)\n                        for index,camera,timestamp,content in parquet_records(local,source,config):\n                            accept(repo,revision,source,index,camera,timestamp,content)\n                            if config[\'limit\'] and processed>=config[\'limit\']: break\n                        flush()\n                        if not config[\'limit\'] or processed<config[\'limit\']:\n                            db.execute(\'INSERT OR IGNORE INTO completed VALUES (?,?,?)\',(repo,revision,source))\n                            db.commit()\n                    if config[\'limit\'] and processed>=config[\'limit\']: break\n            else:\n                if not raw: raise ValueError(f\'{repo}: no raw images; check format/archives\')\n                # Download only one bounded group at a time, then remove its scratch directory.\n                for start in range(0,len(raw),config[\'batch\']):\n                    chunk = raw[start:start+config[\'batch\']]\n                    chunk = [p for p in chunk if not db.execute(\'SELECT 1 FROM counts WHERE id=?\',(\n                        hashlib.sha256(json.dumps([repo,revision,p,0]).encode()).hexdigest(),)).fetchone()]\n                    if config[\'limit\']: chunk=chunk[:max(0,config[\'limit\']-processed)]\n                    with tempfile.TemporaryDirectory(prefix=\'yolo-raw-\') as tmp:\n                        def download(source):\n                            local=hf_hub_download(repo,source,repo_type=\'dataset\',revision=revision,local_dir=tmp)\n                            return Path(local).read_bytes()\n                        with ThreadPoolExecutor(max_workers=config[\'workers\']) as pool:\n                            for source,content in zip(chunk,pool.map(download,chunk)):\n                                camera,timestamp=identity({},source,config[\'timezone\'])\n                                accept(repo,revision,source,0,camera,timestamp,content)\n                        flush()\n                    if config[\'limit\'] and processed>=config[\'limit\']: break\n            export(db,output,names)\n            if config[\'limit\'] and processed>=config[\'limit\']: break\n    finally:\n        db.rollback()  # An interrupted/uncommitted batch is processed again on resume.\n        export(db,output,names)\n        print(\'Saved:\',output,db.execute(\'SELECT status,count(*) FROM counts GROUP BY status\').fetchall())\n        db.close()\n', encoding='utf-8')
import importlib.util
spec = importlib.util.spec_from_file_location('hf_counter', ENGINE)
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)


## Khôi phục checkpoint (tùy chọn)

Nếu phiên trước đã có Output, Add Input của phiên đó, sửa RESUME_FROM rồi chạy cell.
Mỗi lần chỉ một process dùng một SQLite. Khôi phục toàn thư mục vehicle-counts để giữ
config và SQLite; revision từng repo đã pin trong database. Muốn lấy dữ liệu mới từ
repo snapshots5 đang crawl, dùng output mới để tạo snapshot mới rồi kiểm tra trùng khi ghép.


In [ ]:
import shutil
RESUME_FROM = None  # Ví dụ '/kaggle/input/previous-notebook-output/vehicle-counts'
if RESUME_FROM:
    destination = Path(CONFIG['output'])
    assert not (destination / 'counts.sqlite').exists(), 'Output đã tồn tại; không ghi đè checkpoint'
    shutil.copytree(RESUME_FROM, destination, dirs_exist_ok=True)


In [ ]:
# Nếu OOM: giảm batch và chạy lại. Đổi threshold/imgsz/weights cần output mới.
# Để chạy toàn bộ, đặt CONFIG['limit']=None trong cell cấu hình TRƯỚC Save & Run All.
engine.run(CONFIG)


## Xem bounding box và số đếm

Preview là vài ảnh đầu, không đại diện toàn bộ camera/điều kiện. Nên kiểm tra thêm ảnh đêm.
Ảnh lỗi có status=error và số đếm trống; ảnh hợp lệ không có xe có total_vehicles=0.
Camera/time không xác định được sẽ dừng thay vì ghi nhầm dữ liệu.


In [ ]:
import sqlite3
from PIL import Image
from IPython.display import display
out = Path(CONFIG['output'])
with sqlite3.connect(out / 'counts.sqlite') as db:
    print('Thống kê:', db.execute('SELECT repo,status,count(*) FROM counts GROUP BY repo,status').fetchall())
    print('Ví dụ:', db.execute('SELECT camera_id,timestamp,total FROM counts WHERE status="ok" LIMIT 5').fetchall())
for p in sorted((out / 'previews').glob('*.jpg'))[:6]:
    with Image.open(p) as image:
        display(image.copy())


## Lưu Output

CSV có repo, revision, camera, timestamp, tổng xe, số xe theo class, hash ảnh và trạng thái.
Không cộng số đếm các ảnh để suy thành lưu lượng. Trước STGCN cần reindex/resample/missing mask.
Các repo có thể chứa trùng camera/timestamp: CSV giữ nguồn riêng, không tự cộng hoặc chọn một
bản. Khi ghép, kiểm tra image_sha256 và xung đột số đếm.

Checkpoint commit theo batch, export CSV sau mỗi repo và khi thoát qua finally.
Kill cứng có thể chưa có CSV mới nhất, nhưng các batch đã commit ở SQLite có thể resume
nếu Output được giữ lại. Cuối run tạo ZIP; tải cả SQLite để resume, không chỉ CSV.
Ảnh scratch trong /tmp được xóa sau mỗi shard/batch; resume không phụ thuộc mtime ảnh.
Parquet hoàn tất được bỏ qua; shard dở phải tải và quét lại nhưng không inference lại ảnh đã ghi.


In [ ]:
archive = shutil.make_archive('/kaggle/working/vehicle-counts-results', 'zip', CONFIG['output'])
from IPython.display import FileLink
display(FileLink(archive))
